In [1]:
library(ggplot2)
library(dplyr)
library(Seurat)
library(tidyverse)
library(tidyr)
library(stringr)

Warning message:
“package ‘ggplot2’ was built under R version 4.3.3”
Warning message:
“package ‘dplyr’ was built under R version 4.3.2”

Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Warning message:
“package ‘Seurat’ was built under R version 4.3.3”
Loading required package: SeuratObject

Warning message:
“package ‘SeuratObject’ was built under R version 4.3.3”
Loading required package: sp

Warning message:
“package ‘sp’ was built under R version 4.3.2”

Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t


Warning message:
“package ‘tidyverse’ was built under R version 4.3.3”
Warning message:
“package ‘readr’ was built under R version 4.3.3”
Warning message:
“package ‘stringr’ was built under R version 4.3.2”
Warning message:
“package ‘forcats’ was built under R version 4.3.3”


In [ ]:
specific_hyper_hypo_DHMR <- read.csv("./03_Histone/Paired-Tag_DMR/output/00-dhmr_specific_hyper_hypo_DHMR_include_region.csv",row.names=1,check.names=F)

In [3]:
hyper_regions <- specific_hyper_hypo_DHMR %>%
  select(Subclass, Specific_Hyper_Region) %>%
  # Replace single quotes with double quotes
  mutate(Specific_Hyper_Region = str_replace_all(Specific_Hyper_Region, "'", "\"")) %>%
  # Converts a string to a real list
  mutate(Specific_Hyper_Region = map(Specific_Hyper_Region, ~ jsonlite::fromJSON(.))) %>%
  # Expand into separate rows
  unnest(Specific_Hyper_Region) %>%
  rename(region = Specific_Hyper_Region) %>% data.frame()

In [4]:
head(hyper_regions);dim(hyper_regions)

,Subclass,region
,<chr>,<chr>
1,L2/3 IT CTX Glut,chr1_6944000_6944293
2,L2/3 IT CTX Glut,chr1_9781062_9781569
3,L2/3 IT CTX Glut,chr1_20575431_20576126
4,L2/3 IT CTX Glut,chr1_31350695_31351287
5,L2/3 IT CTX Glut,chr1_31956860_31957122
6,L2/3 IT CTX Glut,chr1_35665283_35665889


[1] 105724      2

In [5]:
unique(hyper_regions$Subclass);length(unique(hyper_regions$Subclass))

[1] "L2/3 IT CTX Glut"      "L4/5 IT CTX Glut"      "L5 IT CTX Glut"       
 [4] "L6 IT CTX Glut"        "IT AON-TT-DP Glut"     "LA-BLA-BMA-PA Glut"   
 [7] "L2/3 IT RSP Glut"      "L4 RSP-ACA Glut"       "L5 ET CTX Glut"       
[10] "SUB-ProS Glut"         "CA1-ProS Glut"         "CA3 Glut"             
[13] "CLA-EPd-CTX Car3 Glut" "L5 NP CTX Glut"        "L6 CT CTX Glut"       
[16] "DG Glut"               "OB Eomes Ms4a15 Glut"  "OB-in Frmd7 Gaba"     
[19] "OB-out Frmd7 Gaba"     "Sncg Gaba"             "Lamp5 Gaba"           
[22] "Pvalb Gaba"            "Sst Gaba"              "STR D1 Gaba"          
[25] "STR D2 Gaba"           "ACB-BST-FS D1 Gaba"

[1] 26

In [ ]:
process_regions <- function(subclass1,subclass2) { # 'L2/3 IT CTX Glut', 'L2_3'
  df <- hyper_regions[hyper_regions$Subclass == subclass1,]
  df_split <- do.call(rbind, strsplit(df$region, "_"))
  df_split <- as.data.frame(df_split, stringsAsFactors = FALSE)
  colnames(df_split) <- c("chr", "start", "end")
  df_split$start <- as.numeric(df_split$start)
  df_split$end <- as.numeric(df_split$end)
  df_split$mid <- (df_split$start + df_split$end) / 2
  df_split$new_start <- pmax(0, df_split$mid - 500)
  df_split$new_end <- df_split$mid + 500

  df_final <- data.frame(
    chr = df_split$chr,
    start = as.integer(df_split$new_start),
    end = as.integer(df_split$new_end),
    region = paste0(df_split$chr, ":", as.integer(df_split$new_start), "_", as.integer(df_split$new_end))
  )
  write.table(df_final,
              sprintf("./03_Histone/Paired-Tag_DMR/output/02_DHMR_hyper_bed/%s_hyper_DHMR.bed",subclass2),
              sep = "\t", row.names = FALSE, col.names = FALSE, quote = FALSE)
  print(head(df_final))
  print(dim(df_final))
}

In [7]:
process_regions('L2/3 IT CTX Glut', 'L2_3')

   chr    start      end                 region
1 chr1  6943646  6944646   chr1:6943646_6944646
2 chr1  9780815  9781815   chr1:9780815_9781815
3 chr1 20575278 20576278 chr1:20575278_20576278
4 chr1 31350491 31351491 chr1:31350491_31351491
5 chr1 31956491 31957491 chr1:31956491_31957491
6 chr1 35665086 35666086 chr1:35665086_35666086
[1] 369   4


In [8]:
process_regions('L4/5 IT CTX Glut', 'L4_5')

   chr    start      end                 region
1 chr1  3931138  3932138   chr1:3931138_3932138
2 chr1  7536517  7537517   chr1:7536517_7537517
3 chr1  9794519  9795519   chr1:9794519_9795519
4 chr1 14839073 14840073 chr1:14839073_14840073
5 chr1 14914779 14915779 chr1:14914779_14915779
6 chr1 18962969 18963969 chr1:18962969_18963969
[1] 901   4


In [9]:
process_regions('L5 IT CTX Glut', 'L5')

   chr    start      end                 region
1 chr1  9892852  9893852   chr1:9892852_9893852
2 chr1 11320903 11321903 chr1:11320903_11321903
3 chr1 15848796 15849796 chr1:15848796_15849796
4 chr1 20499848 20500848 chr1:20499848_20500848
5 chr1 22220457 22221457 chr1:22220457_22221457
6 chr1 22315425 22316425 chr1:22315425_22316425
[1] 682   4


In [10]:
process_regions('L5 NP CTX Glut', 'NP')

   chr    start      end                 region
1 chr1  6845070  6846070   chr1:6845070_6846070
2 chr1 11028780 11029780 chr1:11028780_11029780
3 chr1 11614937 11615937 chr1:11614937_11615937
4 chr1 11704039 11705039 chr1:11704039_11705039
5 chr1 12334597 12335597 chr1:12334597_12335597
6 chr1 14916069 14917069 chr1:14916069_14917069
[1] 3378    4


In [11]:
process_regions('L6 CT CTX Glut', 'CT')

   chr    start      end                 region
1 chr1  5417353  5418353   chr1:5417353_5418353
2 chr1  9594969  9595969   chr1:9594969_9595969
3 chr1 10605311 10606311 chr1:10605311_10606311
4 chr1 11728492 11729492 chr1:11728492_11729492
5 chr1 12280996 12281996 chr1:12280996_12281996
6 chr1 14647119 14648119 chr1:14647119_14648119
[1] 970   4


In [12]:
process_regions('DG Glut', 'DG')

   chr    start      end                 region
1 chr1  6862813  6863813   chr1:6862813_6863813
2 chr1 13910825 13911825 chr1:13910825_13911825
3 chr1 16814367 16815367 chr1:16814367_16815367
4 chr1 16867816 16868816 chr1:16867816_16868816
5 chr1 17029988 17030988 chr1:17029988_17030988
6 chr1 20272407 20273407 chr1:20272407_20273407
[1] 676   4


In [13]:
process_regions('CA1-ProS Glut', 'CA1')

   chr   start     end               region
1 chr1 4000851 4001851 chr1:4000851_4001851
2 chr1 4794017 4795017 chr1:4794017_4795017
3 chr1 4947085 4948085 chr1:4947085_4948085
4 chr1 5336791 5337791 chr1:5336791_5337791
5 chr1 5745992 5746992 chr1:5745992_5746992
6 chr1 5825616 5826616 chr1:5825616_5826616
[1] 1183    4


In [14]:
process_regions('CA3 Glut', 'CA23')

   chr   start     end               region
1 chr1 4065207 4066207 chr1:4065207_4066207
2 chr1 4072107 4073107 chr1:4072107_4073107
3 chr1 4118129 4119129 chr1:4118129_4119129
4 chr1 4258756 4259756 chr1:4258756_4259756
5 chr1 4342915 4343915 chr1:4342915_4343915
6 chr1 4354000 4355000 chr1:4354000_4355000
[1] 3038    4


In [15]:
process_regions('Pvalb Gaba', 'Pvalb')

   chr   start     end               region
1 chr1 3006293 3007293 chr1:3006293_3007293
2 chr1 3076588 3077588 chr1:3076588_3077588
3 chr1 3130335 3131335 chr1:3130335_3131335
4 chr1 3152959 3153959 chr1:3152959_3153959
5 chr1 3189048 3190048 chr1:3189048_3190048
6 chr1 3229735 3230735 chr1:3229735_3230735
[1] 12137     4


In [16]:
process_regions('Sst Gaba', 'Sst')

   chr   start     end               region
1 chr1 3035314 3036314 chr1:3035314_3036314
2 chr1 3074178 3075178 chr1:3074178_3075178
3 chr1 3083989 3084989 chr1:3083989_3084989
4 chr1 3098325 3099325 chr1:3098325_3099325
5 chr1 3104126 3105126 chr1:3104126_3105126
6 chr1 3133078 3134078 chr1:3133078_3134078
[1] 14003     4


In [17]:
process_regions('Sncg Gaba', 'Sncg')

   chr    start      end                 region
1 chr1 14733795 14734795 chr1:14733795_14734795
2 chr1 24312129 24313129 chr1:24312129_24313129
3 chr1 24515051 24516051 chr1:24515051_24516051
4 chr1 25016336 25017336 chr1:25016336_25017336
5 chr1 26282610 26283610 chr1:26282610_26283610
6 chr1 32171755 32172755 chr1:32171755_32172755
[1] 711   4


In [18]:
process_regions('Lamp5 Gaba', 'Lamp5')

   chr   start     end               region
1 chr1 3116806 3117806 chr1:3116806_3117806
2 chr1 3290864 3291864 chr1:3290864_3291864
3 chr1 3510333 3511333 chr1:3510333_3511333
4 chr1 3732916 3733916 chr1:3732916_3733916
5 chr1 3889972 3890972 chr1:3889972_3890972
6 chr1 4330502 4331502 chr1:4330502_4331502
[1] 9318    4
